# Rule lab (development only)

Test a JSON Logic rule against a sample before proposing it in the Rule Engine. Proposals still go through
author → review → Compliance approval → shadow/canary in the web app; this notebook never changes rules.

In [ ]:
%glue_version 4.0
%worker_type G.1X
%number_of_workers 2
%idle_timeout 30
%connections lbp-vpc
%additional_python_modules s3://REPLACE-artifacts-bucket/wheels/lbp-1.0.0-py3-none-any.whl
%%configure
{"--LBP_ENV": "aws", "--LBP_DB_SECRET_ARN": "REPLACE", "--LBP_DB_HOST": "REPLACE", "--LBP_PROCESSED_BUCKET": "REPLACE",
 "--LBP_CONFIG_SIGNING_SECRET_ARN": "REPLACE"}

In [ ]:
import os, sys
from awsglue.utils import getResolvedOptions
# Session parameters become LBP_* settings for the lbp package (same settings the jobs use)
for k in ["LBP_ENV", "LBP_DB_SECRET_ARN", "LBP_DB_HOST", "LBP_PROCESSED_BUCKET", "LBP_CONFIG_SIGNING_SECRET_ARN"]:
    try:
        os.environ[k] = getResolvedOptions(sys.argv, [k])[k]
    except Exception:
        pass
os.environ["LBP_AUTO_CREATE_SCHEMA"] = "false"

In [ ]:
FEED_ID = "src_REPLACE"
SAMPLE = "s3://REPLACE-raw/incoming/src_REPLACE/sample.csv"
RULE = {"or": [{"is_blank": {"var": "dti_ratio"}}, {"<=": [{"var": "dti_ratio"}, 45]}]}

In [ ]:
import pandas as pd
from lbp import config_store
from lbp.ingestion.parser import ParseOptions, read_sample
from lbp.mapping.transforms import compile_mapping
from lbp.rules.jsonlogic import apply, validate
from lbp.storage import local_path

print("problems:", validate(RULE) or "none")
cfg = config_store.load(FEED_ID)
meta, rows = read_sample(local_path(SAMPLE), 5000, ParseOptions(**cfg.parse_options))
mapper = compile_mapping(cfg.fields)
recs = [mapper(r)[0] for r in rows]
res = pd.Series([bool(apply(RULE, rec)) for rec in recs])
print(f"{(~res).sum()} of {len(res)} loans would fail ({(~res).mean():.1%})")

In [ ]:
# loans that would fail, with the fields the rule reads (NPI fields are not shown)
from lbp.rules.jsonlogic import referenced_vars
from lbp.ingestion.pii import TARGET_PII_TYPES
cols = [c for c in sorted(referenced_vars(RULE)) if c not in TARGET_PII_TYPES]
pd.DataFrame([{**{"loan_number": r.get("loan_number")}, **{c: r.get(c) for c in cols}} for r, ok in zip(recs, res) if not ok]).head(25)